<a href="https://colab.research.google.com/github/IA2526/trabajo-ia-equipo-4/blob/main/Trabajo_AnalisisDataset_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Grupo 4 - 13/05/2026 - IA - Trabajo IA - Analisis del Dataset

# Notebook 1:  Análisis del Dataset

Análisis del dataset de fotos de actualidad del Gobierno de La Rioja. El objetivo es comprender la estructura de los datos, sus estadísticas básicas y visualizar muestras representativas antes de aplicar cualquier técnica de visión por computador.

## 0. Carga de librerías

Cargamos las librerías necesarias para el análisis.

In [1]:
%%capture
!pip install opencv-contrib-python
!pip install matplotlib

In [2]:
import os
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from collections import Counter
%matplotlib inline

## 1. Carga del Dataset

Para cargar el dataset, lo traeremos de github tal y como ya hicimos en algunos entregables.

In [3]:
%%capture
!wget DATASET_URL -O dados.zip
!unzip dados.zip

## 2. Estructura del dataset

Analizamos cómo está organizado el dataset: carpetas, subcarpetas y número total de imágenes.

In [ ]:
datasetPath = Path('/content/2')

imagenes = [p for p in datasetPath.rglob('*') ]

print(f"Total de imágenes: {len(imagenes)}")
print(f"Carpetas de primer nivel: {[f.name for f in sorted(datasetPath.iterdir()) if f.is_dir()][:10]}")

Total de imágenes: 21678
Carpetas de primer nivel: []


Después de recorrer el dataset vemos que no hay splits previos, simplemente 21.678 imágenes en un único directorio.

## 3. Estadísticas de resolución

Analizamos el alto, ancho y número de canales de todas las imágenes para entender la variabilidad del dataset.

In [ ]:
alturas, anchos  = [], []
fallos = 0
for ruta in imagenes:
    img = cv2.imread(str(ruta))
    #por si no pudo leer bien
    if img is None:
        fallos += 1
        continue
    #info de cada imagen
    h, w = img.shape[:2]
    #Lo guardamos en las listas
    alturas.append(h)
    anchos.append(w)

print(f"Imágenes leídas correctamente: {len(alturas)}")
print(f"Imágenes que no se pudieron leer: {fallos}")

Imágenes leídas correctamente: 21624
Imágenes que no se pudieron leer: 54


In [ ]:
print(f"Tamaño medio de la imagenes: {np.mean(anchos):.0f} x {np.mean(alturas):.0f} px")

topTam = Counter(zip(anchos, alturas)).most_common(5)
print("Tamaños más frecuentes (ancho x alto):")
for (w, h), n in topTam:
    print(f"  {w} x {h} px — {n} imagenes")

bottomTam = Counter(zip(anchos, alturas)).most_common()[:-6:-1]
print("Tamaños menos frecuentes (ancho x alto):")
for (w, h), n in bottomTam:
    print(f"  {w} x {h} px — {n} imagenes")

Tamaño medio de la imagenes: 702 x 499 px
Tamaños más frecuentes (ancho x alto):
  640 x 480 px — 5577 imagenes
  640 x 426 px — 2688 imagenes
  800 x 600 px — 2040 imagenes
  640 x 427 px — 1263 imagenes
  800 x 533 px — 1208 imagenes
Tamaños menos frecuentes (ancho x alto):
  1280 x 918 px — 1 imagenes
  1500 x 1688 px — 1 imagenes
  720 x 440 px — 1 imagenes
  640 x 570 px — 1 imagenes
  800 x 711 px — 1 imagenes


Visualizamos la distribución de resoluciones con un scatter plot para ver si hay agrupaciones claras.

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(anchos, alturas, alpha=0.2, s=8, edgecolors='none')
plt.xlabel('Anchura (px)')
plt.ylabel('Altura (px)')
plt.tight_layout()
plt.show()

<Figure size 800x500 with 1 Axes>

El plot muestra que la gran mayoría de imágenes se concentra en resoluciones bajas , con una densidad muy alta al rededor de 800x600px. Hay un pequeño grupo de imágenes de alta resolución que son claros outliers. Esto nos confirma que las resoluciones del dataset no son homogeneas y que será necesario normalizar las caras a un tamaño fijo en el Notebook 2.

## 4. Visualización del contenido del dataset

Vamos a ver una muestra aleatoria de imágenes del dataset para familiarizarnos con su contenido.

In [ ]:
import random

muestra = random.sample(imagenes, min(12, len(imagenes)))

fig, axes = plt.subplots(3, 4, figsize=(14, 9))
for ax, ruta in zip(axes.flatten(), muestra):
    img = cv2.imread(str(ruta))
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img_rgb)
    ax.axis('off')
plt.suptitle('Muestra aleatoria del dataset', fontsize=13)
plt.tight_layout()
plt.show()

<Figure size 1400x900 with 12 Axes>

## 5. Análisis de tamaño en disco

Comprobamos el peso de las imágenes para estimar el coste computacional del procesado completo.

In [ ]:
tamanios_kb = [os.path.getsize(str(p)) / 1024 for p in imagenes]

print(f"Tamaño medio:  {np.mean(tamanios_kb):.1f} KB")
print(f"Tamaño máximo: {max(tamanios_kb):.1f} KB")
print(f"Tamaño mínimo: {min(tamanios_kb):.1f} KB")
print(f"Peso total:    {sum(tamanios_kb)/1024:.1f} MB")

plt.figure(figsize=(8, 4))
plt.hist(tamanios_kb, bins=40, color='steelblue', edgecolor='white')
plt.xlabel('Tamaño (KB)')
plt.ylabel('Número de imágenes')
plt.title('Distribución del tamaño en disco')
plt.tight_layout()
plt.show()

Tamaño medio:  52.4 KB
Tamaño máximo: 14575.6 KB
Tamaño mínimo: 0.0 KB
Peso total:    1109.8 MB


<Figure size 800x400 with 1 Axes>

El peso medio por imagen es bajo, lo que indica que se trata de fotos pequeñas típicas de web/prensa pero hay algunos outliers que estiran el histograma hasta 14,5 MB. Habra que tener en cuenta esto en un futuro

## 6. Detección inicial de caras

Antes de pasar al Notebook 2, hacemos una pasada rápida sobre todo el dataset usando Haar Cascades para estimar cuántas imágenes tienen al menos una cara detectable.

Usamos el detector `haarcascade_frontalface_default.xml` incluido en [OpenCV](https://pyimagesearch.com/2021/04/05/opencv-face-detection-with-haar-cascades/) con los parámetros recomendados por PyImageSearch: `scaleFactor=1.05`, `minNeighbors=5` y `minSize=(30, 30)`.

In [ ]:
print("[INFO] loading face detector...")
detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

con_cara = 0
sin_cara = 0

for ruta in imagenes:
    image = cv2.imread(str(ruta))
    if image is None:
        continue
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    rects = detector.detectMultiScale(gray, scaleFactor=1.05,
                                      minNeighbors=5, minSize=(30, 30),
                                      flags=cv2.CASCADE_SCALE_IMAGE)
    if len(rects) > 0:
        con_cara += 1
    else:
        sin_cara += 1

print("[INFO] {} imágenes con al menos una cara detectada ({:.1f}%)".format(
    con_cara, 100 * con_cara / (con_cara + sin_cara)))
print("[INFO] {} imágenes sin cara detectada ({:.1f}%)".format(
    sin_cara, 100 * sin_cara / (con_cara + sin_cara)))

[INFO] loading face detector...


Se ha ejecutado esta celda durante 1h y 40 y no da resultado en colab por lo que decidimos ejecutarlo en local aplicando procesamiento paralelo mediante múltiples procesos.

In [ ]:
import multiprocessing
import concurrent.futures
from pathlib import Path

In [ ]:
def detectar_cara(ruta):
    detector = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    image = cv2.imread(str(ruta))
    if image is None:
        return 0
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    rects = detector.detectMultiScale(gray, scaleFactor=1.05,
                                      minNeighbors=5, minSize=(30, 30),
                                      flags=cv2.CASCADE_SCALE_IMAGE)
    return 1 if len(rects) > 0 else 0

In [ ]:
datasetPath = Path('noticiasFotos/2')

imagenes = [p for p in datasetPath.rglob('*') ]

print(f"Total de imágenes: {len(imagenes)}")
print(f"Carpetas de primer nivel: {[f.name for f in sorted(datasetPath.iterdir()) if f.is_dir()][:10]}")

Total de imágenes: 21678
Carpetas de primer nivel: []


In [ ]:
with concurrent.futures.ThreadPoolExecutor() as executor:
    resultados = list(executor.map(detectar_cara, imagenes))

con_cara = sum(resultados)
sin_cara = len(resultados) - con_cara

print(f"[INFO] Con cara: {con_cara} ({100 * con_cara / len(resultados):.1f}%)")
print(f"[INFO] Sin cara: {sin_cara} ({100 * sin_cara / len(resultados):.1f}%)")

[INFO] Con cara: 17809 (82.2%)
[INFO] Sin cara: 3869 (17.8%)


El análisis sobre las 21.678 imágenes, ejecutado en local tardo tan solo 6 minutos. Vemos que el 82.2% contienen al menos una cara detectable con Haar Cascades.

Hay que destacar que esta detección con Haar Cascades es una aproximación superficial para comprobar si el dataset es propenso a contener caras y, por tanto, si tiene sentido aplicar un motor de búsqueda facial sobre él. El resultado del 82,2% es bueno, pero hay que tomar este dato pinzas ya que Haar Cascades es un detector rápido pero poco preciso, con tendencia a generar falsos positivos y a perder caras que estan de perfil, con oclusiones o en baja resolución y por tanto el número real de caras válidas podría variar.

## 7. Conclusiones

El análisis nos ha permitido conocer en profundidad la estructura del dataset. En este hay 21.678 imágenes sin splits previos, resoluciones heterogéneas concentradas en torno a 800×600 px y un 82,2% de imágenes con al menos una cara detectable.

Con esta base, en el Notebook 02 haremos la detección y recorte de caras sobre todo el dataset de manera mas rigurosa, utilizando un detector más robusto que Haar Cascades y finalmente la normalizaremos cada cara a un tamaño fijo.